# 25 Models: the kinds of outcome, links, tables of tests, predictions

Real-data checks of `table.explain` and its neighbours in the group "Models". This notebook holds only simplydata:
it reads the data with `sdata.table`, runs the words, reads the numbers back with simplydata's own words
(`get_column`, `get_row`, `get_cell_at`, through `Result`) and compares them with `references/25_models_outcomes.csv`.
Each comparison prints its "Source:" with a link.

The references are, first, numbers printed in the documentation of the library simplydata calls or in the
literature, typed in as printed and compared to half a unit of their last digit: Zeileis, Kleiber & Jackman (2008)
for the NMES 1988 count models (the pscl vignette); Dobson & Barnett (2008) for the probit and complementary log-log
fits of Bliss's beetles; Ferrari & Cribari-Neto (2004) Table 1, as the betareg vignette prints it; McCullagh &
Nelder (1989) for the clotting Gamma fit; the quantreg vignette for Engel's median regression; the statsmodels RLM
docstring for the stack loss data. Only where nothing is printed, `scripts/references/25_models_outcomes.py` runs the
help-page example of the library once (R glm/sandwich, pscl, confint, car, effectsize, MASS, nnet, quantreg, predict)
and saves its value with the link to that page (tolerance relative 1e-10 for closed formulas, 1e-6 for fits by
iteration). It also saves the R datasets this notebook reads.

| section | word and option | data |
|---|---|---|
| 1 | `explain(outcome="counts" / "counts (overdispersed)" / "counts that vary a lot" / "counts in two steps that vary a lot" / "counts with many zeros that vary a lot", zero_part_using=, robust_errors=True)` | NMES 1988 |
| 2 | `explain(outcome="counts with many zeros")` | bioChemists (pscl) |
| 3 | `explain(outcome="yes or no", link="probit" / "cloglog", interval_by="profile")` | Bliss's beetles |
| 4 | `explain(outcome="shares")` | GasolineYield (betareg) |
| 5 | `explain(outcome="positive numbers", link="inverse")` | clotting times |
| 6 | `explain(table_of_tests="type 2" / "type 3" / "in order")` | Moore (carData) |
| 7 | `explain(outcome="ordered groups" / "groups")` | housing (MASS) |
| 8 | `explain(for_percentile=)` | engel (quantreg) |
| 9 | `explain(giving_outliers_less_weight=True)` | stackloss |
| 10 | `explain(show="standardised effects", weights=)` | stackloss, Bliss's beetles |
| 11 | `add_predicted_column(outcome=, interval=, for_new_rows=, weights=)` | stackloss, birthwt, NMES 1988, beetles |

Section 12: the rows `show_rows_that_pull_model` gives back, and the numbers `check_model_of(show="numbers")` gives back
(both from simplydata 3fd3b85 on; before, they were only printed and these checks waited for those words).

In [1]:
import simplydata as sdata

from reference_checks import Comparisons, Result, quiet

cmp = Comparisons("25_models_outcomes")
DATA = "../data/"
FETCHED = "../data/fetched/"  # the datasets this repository does not host (written by scripts/get_data.py)

## 1. Counts: NMES 1988 and the models of Zeileis, Kleiber & Jackman (2008)

Deb & Trivedi's data on 4406 people aged 66 and over: physician office visits explained by hospital stays, health,
chronic conditions, gender, years of education and private insurance (`data/fetched/nmes1988.csv`; visits = ofp, hospital =
hosp, chronic = numchron, insurance = privins). simplydata's baselines (the first group in sorted order: health
"average", gender "female", insurance "no") are the article's.

In [2]:
nmes = sdata.table(FETCHED + "nmes1988.csv", separator=",")
USING = ["hospital", "health", "chronic", "gender", "school", "insurance"]
nmes.explain("visits", using=USING, outcome="counts")

Explaining "visits" from "hospital", "health", "chronic", "gender", "school" and "insurance" (4406 rows). Each rate ratio below holds the other columns fixed:
  intercept: 2.80, the expected count when every number column is 0 and every group column is at its baseline.
  hospital: each 1 more "hospital" multiplies the expected count by 1.18 (95 % CI 1.17 to 1.19); fewer than 1 in 1000 studies would show this much if there were truly none (p < 0.0001).
  health "excellent" against "average": the expected count is 0.696 times as high (95 % CI 0.656 to 0.739); fewer than 1 in 1000 studies would show this much if there were truly none (p < 0.0001).
  health "poor" against "average": the expected count is 1.28 times as high (95 % CI 1.24 to 1.33); fewer than 1 in 1000 studies would show this much if there were truly none (p < 0.0001).
  chronic: each 1 more "chronic" multiplies the expected count by 1.16 (95 % CI 1.15 to 1.17); fewer than 1 in 1000 studies would show this much if there were

,term,rate ratio,95 % CI low,95 % CI high,log rate ratio,standard error,statistic name,statistic,degrees of freedom,p-value
1,(intercept),2.797914,2.670476,2.931434,1.028874,0.023785,z,43.257469,NaN,0.000000e+00
2,hospital,1.179154,1.165375,1.193097,0.164797,0.005997,z,27.478180,NaN,3.200884e-166
3,health (excellent),0.696287,0.656135,0.738896,-0.361993,0.030304,z,-11.945234,NaN,6.875623e-33
4,health (poor),1.281853,1.237796,1.327479,0.248307,0.017845,z,13.914926,NaN,5.141310e-44
5,chronic,1.157936,1.147589,1.168377,0.146639,0.004580,z,32.019426,NaN,5.851670e-225
6,gender (male),0.893758,0.871367,0.916725,-0.112320,0.012945,z,-8.676534,NaN,4.080121e-18
7,school,1.026488,1.022786,1.030203,0.026143,0.001843,z,14.182368,NaN,1.178033e-45
8,insurance (yes),1.223465,1.183696,1.264570,0.201687,0.016860,z,11.962403,NaN,5.591976e-33
9,health (whole column),NaN,NaN,NaN,NaN,NaN,chi-square,359.478497,2.0,8.714376e-79


In [3]:
TERMS = ["(intercept)", "hospital", "health (poor)", "health (excellent)", "chronic", "gender (male)", "school", "insurance (yes)"]
poisson = Result(quiet(nmes.explain, "visits", using=USING, outcome="counts"), key="term")
for term in TERMS:
    cmp.check("NMES 1988", 'explain (outcome="counts")', f"{term} estimate", poisson[term, "log rate ratio"])
    cmp.check("NMES 1988", 'explain (outcome="counts")', f"{term} standard error", poisson[term, "standard error"])

robust = Result(quiet(nmes.explain, "visits", using=USING, outcome="counts", robust_errors=True), key="term")
for term in TERMS:
    cmp.check("NMES 1988", 'explain (outcome="counts", robust_errors=True)', f"{term} standard error (HC3)", robust[term, "standard error"])

# quasi-Poisson: the article prints the dispersion (6.706); with the same estimates it is (quasi SE / Poisson SE) squared
quasi = Result(quiet(nmes.explain, "visits", using=USING, outcome="counts (overdispersed)"), key="term")
ratios = [(quasi[t, "standard error"] / poisson[t, "standard error"]) ** 2 for t in TERMS]
cmp.check("NMES 1988", 'explain (outcome="counts (overdispersed)")', "dispersion", sum(ratios) / len(ratios))
for term in TERMS:
    cmp.check("NMES 1988", 'explain (outcome="counts (overdispersed)")', f"{term} standard error", quasi[term, "standard error"])

nb = Result(quiet(nmes.explain, "visits", using=USING, outcome="counts that vary a lot"), key="term")
for term in TERMS:
    cmp.check("NMES 1988", 'explain (outcome="counts that vary a lot")', f"{term} estimate", nb[term, "log rate ratio"])
    cmp.check("NMES 1988", 'explain (outcome="counts that vary a lot")', f"{term} standard error", nb[term, "standard error"])

hurdle = quiet(nmes.explain, "visits", using=USING, outcome="counts in two steps that vary a lot")
for part in ("count", "more than 0"):
    h = Result(hurdle, key="term", where={"part": part})
    for term in TERMS:
        cmp.check("NMES 1988", 'explain (outcome="counts in two steps that vary a lot")', f"{part}: {term} estimate", h[term, "log ratio"])
        cmp.check("NMES 1988", 'explain (outcome="counts in two steps that vary a lot")', f"{part}: {term} standard error",
                  h[term, "standard error"])

zinb = quiet(nmes.explain, "visits", using=USING, outcome="counts with many zeros that vary a lot",
             zero_part_using=["hospital", "chronic", "insurance", "school", "gender"])
CHECK = 'explain (outcome="counts with many zeros that vary a lot", zero_part_using=)'
for part, terms in (("count", TERMS), ("extra zeros", ["(intercept)", "hospital", "chronic", "insurance (yes)", "school", "gender (male)"])):
    z = Result(zinb, key="term", where={"part": part})
    for term in terms:
        cmp.check("NMES 1988", CHECK, f"{part}: {term} estimate", z[term, "log ratio"])
        cmp.check("NMES 1988", CHECK, f"{part}: {term} standard error", z[term, "standard error"])

agrees  NMES 1988 | explain (outcome="counts") | (intercept) estimate: simplydata 1.02887419508, Zeileis, Kleiber & Jackman (2008), J. Stat. Softw. 27(8), summary(fm_pois) 1.028874; difference 1.95e-07 (allowed 5e-07)
      Source: Zeileis, Kleiber & Jackman (2008), J. Stat. Softw. 27(8), summary(fm_pois) (printed) - https://doi.org/10.18637/jss.v027.i08 ; https://cran.r-project.org/web/packages/pscl/vignettes/countreg.pdf
agrees  NMES 1988 | explain (outcome="counts") | (intercept) standard error: simplydata 0.0237848912619, Zeileis, Kleiber & Jackman (2008), J. Stat. Softw. 27(8), summary(fm_pois) 0.023785; difference 1.09e-07 (allowed 5e-07)
      Source: Zeileis, Kleiber & Jackman (2008), J. Stat. Softw. 27(8), summary(fm_pois) (printed) - https://doi.org/10.18637/jss.v027.i08 ; https://cran.r-project.org/web/packages/pscl/vignettes/countreg.pdf
agrees  NMES 1988 | explain (outcome="counts") | hospital estimate: simplydata 0.164797389209, Zeileis, Kleiber & Jackman (2008), J. Stat.

agrees  NMES 1988 | explain (outcome="counts", robust_errors=True) | (intercept) standard error (HC3): simplydata 0.0648967250891, R sandwich vcovHC (type HC3) of the article's Poisson glm (sandwich 3.1.3) 0.0648967250891; relative difference 3.42e-15 (allowed 1e-06)
      Source: R sandwich vcovHC (type HC3) of the article's Poisson glm (sandwich 3.1.3) (run reference) - https://search.r-project.org/CRAN/refmans/sandwich/html/vcovHC.html
agrees  NMES 1988 | explain (outcome="counts", robust_errors=True) | hospital standard error (HC3): simplydata 0.0226830104407, R sandwich vcovHC (type HC3) of the article's Poisson glm (sandwich 3.1.3) 0.0226830104407; relative difference 3.98e-15 (allowed 1e-06)
      Source: R sandwich vcovHC (type HC3) of the article's Poisson glm (sandwich 3.1.3) (run reference) - https://search.r-project.org/CRAN/refmans/sandwich/html/vcovHC.html
agrees  NMES 1988 | explain (outcome="counts", robust_errors=True) | health (poor) standard error (HC3): simplydata 0

agrees  NMES 1988 | explain (outcome="counts in two steps that vary a lot") | count: (intercept) estimate: simplydata 1.1976989152, Zeileis, Kleiber & Jackman (2008), J. Stat. Softw. 27(8), summary(fm_hurdle0) 1.197699; difference 8.48e-08 (allowed 5e-07)
      Source: Zeileis, Kleiber & Jackman (2008), J. Stat. Softw. 27(8), summary(fm_hurdle0) (printed) - https://doi.org/10.18637/jss.v027.i08 ; https://cran.r-project.org/web/packages/pscl/vignettes/countreg.pdf
agrees  NMES 1988 | explain (outcome="counts in two steps that vary a lot") | count: (intercept) standard error: simplydata 0.0589734932489, Zeileis, Kleiber & Jackman (2008), J. Stat. Softw. 27(8), summary(fm_hurdle0) 0.058973; difference 4.93e-07 (allowed 5e-07)
      Source: Zeileis, Kleiber & Jackman (2008), J. Stat. Softw. 27(8), summary(fm_hurdle0) (printed) - https://doi.org/10.18637/jss.v027.i08 ; https://cran.r-project.org/web/packages/pscl/vignettes/countreg.pdf
agrees  NMES 1988 | explain (outcome="counts in two ste

agrees  NMES 1988 | explain (outcome="counts with many zeros that vary a lot", zero_part_using=) | count: (intercept) estimate: simplydata 1.19371555268, Zeileis, Kleiber & Jackman (2008), J. Stat. Softw. 27(8), Table 2 (fm_zinb) 1.194; difference 2.84e-04 (allowed 5e-04)
      Source: Zeileis, Kleiber & Jackman (2008), J. Stat. Softw. 27(8), Table 2 (fm_zinb) (printed) - https://doi.org/10.18637/jss.v027.i08 ; https://cran.r-project.org/web/packages/pscl/vignettes/countreg.pdf
agrees  NMES 1988 | explain (outcome="counts with many zeros that vary a lot", zero_part_using=) | count: (intercept) standard error: simplydata 0.0566608413762, Zeileis, Kleiber & Jackman (2008), J. Stat. Softw. 27(8), Table 2 (fm_zinb) 0.057; difference 3.39e-04 (allowed 5e-04)
      Source: Zeileis, Kleiber & Jackman (2008), J. Stat. Softw. 27(8), Table 2 (fm_zinb) (printed) - https://doi.org/10.18637/jss.v027.i08 ; https://cran.r-project.org/web/packages/pscl/vignettes/countreg.pdf
agrees  NMES 1988 | explai

## 2. Zero-inflated Poisson: the pscl help-page example on bioChemists

`zeroinfl(art ~ ., data = bioChemists)` (Long's 1990 data on 915 biochemists), run once in the reference script.

In [4]:
bio = quiet(sdata.table(cmp.data_file("bioChemists")).explain, "art", using=["fem", "mar", "kid5", "phd", "ment"],
            outcome="counts with many zeros")
for part in ("count", "extra zeros"):
    z = Result(bio, key="term", where={"part": part})
    for term in ["(intercept)", "fem (Women)", "mar (Single)", "kid5", "phd", "ment"]:
        for column in ("log ratio", "standard error", "p-value"):
            cmp.check("bioChemists (pscl)", 'explain (outcome="counts with many zeros")', f"{part}: {term} {column}", z[term, column])

Read "bioChemists.csv" with "," between the columns (guessed). To be sure, give separator=",".
agrees  bioChemists (pscl) | explain (outcome="counts with many zeros") | count: (intercept) log ratio: simplydata 0.744588265406, R pscl zeroinfl (help page example) (pscl 1.5.9) 0.744588265406; relative difference 1.49e-16 (allowed 1e-06)
      Source: R pscl zeroinfl (help page example) (pscl 1.5.9) (run reference) - https://search.r-project.org/CRAN/refmans/pscl/html/zeroinfl.html
agrees  bioChemists (pscl) | explain (outcome="counts with many zeros") | count: (intercept) standard error: simplydata 0.110281004406, R pscl zeroinfl (help page example) (pscl 1.5.9) 0.110281004406; relative difference 9.94e-15 (allowed 1e-06)
      Source: R pscl zeroinfl (help page example) (pscl 1.5.9) (run reference) - https://search.r-project.org/CRAN/refmans/pscl/html/zeroinfl.html
agrees  bioChemists (pscl) | explain (outcome="counts with many zeros") | count: (intercept) p-value: simplydata 1.460861723

## 3. Probit and complementary log-log links; profile intervals: Bliss's beetles

Dobson & Barnett (2008, Table 7.3) print the probit fit -34.935 (2.648) + 19.728 (1.487) x log10 dose and the
complementary log-log fit -39.572 (3.240) + 22.041 (1.799). `interval_by="profile"`: R `confint()` of the logistic glm.

In [5]:
beetles = sdata.table(DATA + "bliss.csv")
beetles.explain("killed", using=["log10_dose"], outcome="yes or no", out_of="beetles", link="probit")

Read "bliss.csv" with "," between the columns (guessed). To be sure, give separator=",".
Explaining "killed" from "log10_dose" (8 rows). Each effect (probit scale) below holds the other columns fixed:
  intercept: -34.9, on the probit scale, when every number column is 0.
  log10_dose: each 1 more "log10_dose" changes the probit of yes by 19.7 (95 % CI 16.8 to 22.6); fewer than 1 in 1000 studies would show this much if there were truly none (p < 0.0001).
Method: probit regression, as R glm(cbind(killed, beetles - killed) ~ log10_dose, family = binomial(link = "probit")); counted as yes: the number in "killed" out of "beetles" tries. Wald 95 % intervals and z-tests, as R confint.default() and summary().
Residual deviance 10.1 on 6 df (with no columns: 284 on 7 df). The model as a whole (likelihood-ratio chi-square = 274, df = 1): fewer than 1 in 1000 studies would show this much if none of the columns truly mattered (p < 0.0001).
How well it fits: pseudo R squared 0.883 (0 = no better t

,term,effect (probit scale),95 % CI low,95 % CI high,standard error,statistic name,statistic,degrees of freedom,p-value
1,(intercept),-34.935259,-40.125081,-29.745436,2.647917,z,-13.193485,NaN,9.565861e-40
2,log10_dose,19.727934,16.813007,22.642861,1.487235,z,13.264842,NaN,3.702026e-40


In [6]:
for link in ("probit", "cloglog"):
    fit = Result(quiet(beetles.explain, "killed", using=["log10_dose"], outcome="yes or no", out_of="beetles", link=link), key="term")
    effect = f"effect ({link} scale)"
    cmp.check_all("Bliss beetles", f'explain (outcome="yes or no", out_of=, link="{link}")',
                  {"intercept": fit["(intercept)", effect], "intercept standard error": fit["(intercept)", "standard error"],
                   "slope": fit["log10_dose", effect], "slope standard error": fit["log10_dose", "standard error"]})
# the interval of the log odds ratio: simplydata gives the odds-ratio interval and the log odds ratio; the profile
# interval of the coefficient is read from the log of the odds-ratio ends with simplydata's own maths word
profile = quiet(beetles.explain, "killed", using=["log10_dose"], outcome="yes or no", out_of="beetles", interval_by="profile")
ends = Result(profile, key="term")
logs = sdata.numbers([ends[t, c] for t in ("(intercept)", "log10_dose") for c in ("95 % CI low", "95 % CI high")]).take_log()
cmp.check_all("Bliss beetles", 'explain (outcome="yes or no", out_of=, interval_by="profile")',
              dict(zip(["intercept 95 % low", "intercept 95 % high", "slope 95 % low", "slope 95 % high"], list(logs))))

agrees  Bliss beetles | explain (outcome="yes or no", out_of=, link="probit") | intercept: simplydata -34.9352586656, Dobson & Barnett (2008), An Introduction to Generalized Linear Models, 3rd ed., Table 7.3 -34.935; difference 2.59e-04 (allowed 5e-04)
      Source: Dobson & Barnett (2008), An Introduction to Generalized Linear Models, 3rd ed., Table 7.3 (printed) - https://doi.org/10.1201/9780367807849
agrees  Bliss beetles | explain (outcome="yes or no", out_of=, link="probit") | intercept standard error: simplydata 2.6479174017, Dobson & Barnett (2008), An Introduction to Generalized Linear Models, 3rd ed., Table 7.3 2.648; difference 8.26e-05 (allowed 5e-04)
      Source: Dobson & Barnett (2008), An Introduction to Generalized Linear Models, 3rd ed., Table 7.3 (printed) - https://doi.org/10.1201/9780367807849
agrees  Bliss beetles | explain (outcome="yes or no", out_of=, link="probit") | slope: simplydata 19.7279340772, Dobson & Barnett (2008), An Introduction to Generalized Linear

## 4. Beta regression for shares: Prater's gasoline yields

The betareg vignette prints Ferrari & Cribari-Neto's (2004) Table 1, with batch 10 as R's baseline. simplydata compares
each batch with batch "1" (the first in sorted order): R's batch k effect is simplydata's batch k minus its batch 10,
R's intercept is simplydata's intercept plus its batch 10, and R's batch 1 is simplydata's batch 10 with the sign turned.

In [7]:
gasoline = sdata.table(cmp.data_file("GasolineYield"))
quiet(gasoline.turn_into_text, "batch")  # the batch numbers are names of groups
beta = Result(quiet(gasoline.explain, "yield", using=["batch", "temp"], outcome="shares"), key="term")
b10 = beta["batch (10)", "log odds ratio"]
ours = {"intercept (batch 10 coding)": beta["(intercept)", "log odds ratio"] + b10, "batch 1 (batch 10 coding)": -b10,
        "batch 1 standard error": beta["batch (10)", "standard error"], "temp": beta["temp", "log odds ratio"],
        "temp standard error": beta["temp", "standard error"]}
ours.update({f"batch {k} (batch 10 coding)": beta[f"batch ({k})", "log odds ratio"] - b10 for k in range(2, 10)})
cmp.check_all("GasolineYield (betareg)", 'explain (outcome="shares")', ours)

Read "GasolineYield.csv" with "," between the columns (guessed). To be sure, give separator=",".


agrees  GasolineYield (betareg) | explain (outcome="shares") | intercept (batch 10 coding): simplydata -6.159571047, Ferrari & Cribari-Neto (2004) Table 1, printed in the betareg vignette (Cribari-Neto & Zeileis 2010) -6.159571; difference 4.70e-08 (allowed 5e-07)
      Source: Ferrari & Cribari-Neto (2004) Table 1, printed in the betareg vignette (Cribari-Neto & Zeileis 2010) (printed) - https://doi.org/10.18637/jss.v034.i02 ; https://cran.r-project.org/web/packages/betareg/vignettes/betareg.html
agrees  GasolineYield (betareg) | explain (outcome="shares") | batch 1 (batch 10 coding): simplydata 1.72772887507, Ferrari & Cribari-Neto (2004) Table 1, printed in the betareg vignette (Cribari-Neto & Zeileis 2010) 1.727729; difference 1.25e-07 (allowed 5e-07)
      Source: Ferrari & Cribari-Neto (2004) Table 1, printed in the betareg vignette (Cribari-Neto & Zeileis 2010) (printed) - https://doi.org/10.18637/jss.v034.i02 ; https://cran.r-project.org/web/packages/betareg/vignettes/betareg.h

## 5. Positive numbers (Gamma regression): blood clotting times

The nine clotting times of the R `glm` help page (McCullagh & Nelder 1989, section 8.4.2), typed in as printed there.
McCullagh & Nelder give 1/mu = -0.01655 + 0.01534 log(u) (`link="inverse"`); the default log link against the help
page's call run once.

In [8]:
clotting = sdata.table({"u": [5, 10, 15, 20, 30, 40, 60, 80, 100], "lot1": [118, 58, 42, 35, 27, 25, 21, 19, 18]})
quiet(clotting.add_column, "log_u", "log(u)")
inverse = Result(quiet(clotting.explain, "lot1", using=["log_u"], outcome="positive numbers", link="inverse"), key="term")
cmp.check_all("clotting times (R glm help page)", 'explain (outcome="positive numbers", link="inverse")',
              {"intercept": inverse["(intercept)", "effect (inverse scale)"], "slope": inverse["log_u", "effect (inverse scale)"]})
log_link = Result(quiet(clotting.explain, "lot1", using=["log_u"], outcome="positive numbers"), key="term")
value = [c for c in log_link.columns if c.startswith("log ")][0]  # the coefficient on the log scale
cmp.check_all("clotting times (R glm help page)", 'explain (outcome="positive numbers")',
              {"intercept": log_link["(intercept)", value], "slope": log_link["log_u", value],
               "intercept standard error": log_link["(intercept)", "standard error"],
               "slope standard error": log_link["log_u", "standard error"], "slope p-value": log_link["log_u", "p-value"]})

agrees  clotting times (R glm help page) | explain (outcome="positive numbers", link="inverse") | intercept: simplydata -0.0165543817262, McCullagh & Nelder (1989), Generalized Linear Models, 2nd ed., section 8.4.2 -0.01655; difference 4.38e-06 (allowed 5e-06)
      Source: McCullagh & Nelder (1989), Generalized Linear Models, 2nd ed., section 8.4.2 (printed) - https://doi.org/10.1201/9780203753736
agrees  clotting times (R glm help page) | explain (outcome="positive numbers", link="inverse") | slope: simplydata 0.0153431149103, McCullagh & Nelder (1989), Generalized Linear Models, 2nd ed., section 8.4.2 0.01534; difference 3.11e-06 (allowed 5e-06)
      Source: McCullagh & Nelder (1989), Generalized Linear Models, 2nd ed., section 8.4.2 (printed) - https://doi.org/10.1201/9780203753736
agrees  clotting times (R glm help page) | explain (outcome="positive numbers") | intercept: simplydata 5.50323022752, R stats glm (help page clotting example, Gamma with log link) (stats 4.6.2) 5.50322

## 6. Tables of tests: type 2, type 3 and in order (Moore's conformity data)

The help-page example of `car::Anova` (types 2 and 3, sum-to-zero contrasts) and `anova()` (in order), with partial
eta-squared from `effectsize::eta_squared`, run once.

In [9]:
moore = sdata.table(cmp.data_file("Moore"))
moore.explain("conformity", using=["fcategory * partner.status"], table_of_tests="type 3")

Read "Moore.csv" with "," between the columns (guessed). To be sure, give separator=",".
Explaining "conformity" from "fcategory" and "partner.status" (45 rows). Each effect below holds the other columns fixed:
  intercept: 11.9, the expected "conformity" when every number column is 0 and every group column is at its baseline.
  fcategory "low" against "high": "conformity" is 5.54 higher (95 % CI 0.120 to 11.0); about 5 of 100 studies would show this much if there were truly none (p = 0.045).
  fcategory "medium" against "high": "conformity" is 2.42 higher (95 % CI -2.06 to 6.89); about 28 of 100 studies would show this much if there were truly none (p = 0.282).
  partner.status "low" against "high": "conformity" is 0.768 higher (95 % CI -4.03 to 5.56); about 75 of 100 studies would show this much if there were truly none (p = 0.748).
  fcategory (low) : partner.status (low) (interaction): -9.27 (95 % CI -16.2 to -2.29), how much the effect of one changes with the other; about 1 of 100

Tests for each column (type 3: each column after all the others, with sum-to-zero coding, as R car::Anova(type = 3)):
  fcategory: F = 0.859, df = 2 and 39, partial eta-squared 0.0422; about 43 of 100 studies would show this much if this column truly did not matter (p = 0.431).
  partner.status: F = 11.4, df = 1 and 39, partial eta-squared 0.227; about 2 of 1000 studies would show this much if this column truly did not matter (p = 0.002).
  fcategory : partner.status: F = 4.18, df = 2 and 39, partial eta-squared 0.177; about 2 of 100 studies would show this much if this column truly did not matter (p = 0.023).
Partial eta-squared: the share of the variation in the outcome that a column explains once the other columns are set aside (R effectsize::eta_squared(partial = TRUE)).


,term,effect,95 % CI low,95 % CI high,standard error,statistic name,statistic,degrees of freedom,p-value,sum of squares,partial eta-squared
1,(intercept),11.857143,8.356384,15.357902,1.730743,t,6.850896,39.0,3.435671e-08,NaN,NaN
2,fcategory (low),5.542857,0.119505,10.966209,2.681256,t,2.067261,39.0,4.539152e-02,NaN,NaN
3,fcategory (medium),2.415584,-2.062601,6.893770,2.213974,t,1.091063,39.0,2.819404e-01,NaN,NaN
4,partner.status (low),0.767857,-4.025754,5.561469,2.369918,t,0.324002,39.0,7.476681e-01,NaN,NaN
5,fcategory (low) : partner.status (low),-9.267857,-16.247461,-2.288254,3.450653,t,-2.685827,39.0,1.057284e-02,NaN,NaN
6,fcategory (medium) : partner.status (low),-7.790584,-15.017225,-0.563943,3.572786,t,-2.180535,39.0,3.531472e-02,NaN,NaN
7,fcategory : partner.status (whole column),NaN,NaN,NaN,NaN,F,4.184623,2.0,2.257244e-02,NaN,NaN
8,"fcategory (test, type 3)",NaN,NaN,NaN,NaN,F,0.858884,2.0,4.314916e-01,36.018706,0.042187
9,"partner.status (test, type 3)",NaN,NaN,NaN,NaN,F,11.424975,1.0,1.657113e-03,239.562370,0.226574
10,"fcategory : partner.status (test, type 3)",NaN,NaN,NaN,NaN,F,4.184623,2.0,2.257244e-02,175.488928,0.176681


In [10]:
for kind in ("type 2", "type 3", "in order"):
    t = Result(quiet(moore.explain, "conformity", using=["fcategory * partner.status"], table_of_tests=kind), key="term")
    for effect in ["fcategory", "partner.status", "fcategory : partner.status"]:
        row = f"{effect} (test, {kind})"
        cmp.check_all("Moore (carData)", f'explain (table_of_tests="{kind}")',
                      {f"{effect}: sum of squares": t[row, "sum of squares"], f"{effect}: F": t[row, "statistic"],
                       f"{effect}: degrees of freedom": t[row, "degrees of freedom"], f"{effect}: p-value": t[row, "p-value"],
                       f"{effect}: partial eta-squared": t[row, "partial eta-squared"]})

agrees  Moore (carData) | explain (table_of_tests="type 2") | fcategory: sum of squares: simplydata 11.6147000439, R car::Anova (help page example) (car 3.1.5) 11.6147000439; relative difference 1.70e-14 (allowed 1e-10)
      Source: R car::Anova (help page example) (car 3.1.5) (run reference) - https://search.r-project.org/CRAN/refmans/car/html/Anova.html
agrees  Moore (carData) | explain (table_of_tests="type 2") | fcategory: F: simplydata 0.276958464358, R car::Anova (help page example) (car 3.1.5) 0.276958464358; relative difference 1.72e-14 (allowed 1e-10)
      Source: R car::Anova (help page example) (car 3.1.5) (run reference) - https://search.r-project.org/CRAN/refmans/car/html/Anova.html
agrees  Moore (carData) | explain (table_of_tests="type 2") | fcategory: degrees of freedom: simplydata 2, R car::Anova (help page example) (car 3.1.5) 2; relative difference 0.00e+00 (allowed 1e-10)
      Source: R car::Anova (help page example) (car 3.1.5) (run reference) - https://search.r

## 7. Ordered groups and groups: Copenhagen housing satisfaction

The help-page example of `MASS::polr` (`weights = Freq`) and `nnet::multinom` on the same model. simplydata has no
frequency weights for these outcomes, so the reference script saves the table with one row per tenant (the same
likelihood). Baselines: Infl "High", Type "Apartment", Cont "High" (first in sorted order; R is given the same).

In [11]:
housing = sdata.table(cmp.data_file("housing_one_row_per_tenant"))
PO = ["Infl (Low)", "Infl (Medium)", "Type (Atrium)", "Type (Terrace)", "Type (Tower)", "Cont (Low)"]
po = Result(quiet(housing.explain, "Sat", using=["Infl", "Type", "Cont"], outcome="ordered groups", in_order=["Low", "Medium", "High"]),
            key="term")
ours = {f"{t} estimate": po[t, "log odds ratio"] for t in PO}
ours.update({f"{t} standard error": po[t, "standard error"] for t in PO})
ours.update({c: po[c, "log odds ratio"] for c in ["cut-point Low | Medium", "cut-point Medium | High"]})
cmp.check_all("housing (MASS)", 'explain (outcome="ordered groups", in_order=)', ours)
mn = quiet(housing.explain, "Sat", using=["Infl", "Type", "Cont"], outcome="groups")
for group in ("Low", "Medium"):
    g = Result(mn, key="term", where={"group": group})
    for t in ["(intercept)"] + PO:
        cmp.check("housing (MASS)", 'explain (outcome="groups")', f"{group}: {t} estimate", g[t, "log odds ratio"])
        cmp.check("housing (MASS)", 'explain (outcome="groups")', f"{group}: {t} standard error", g[t, "standard error"])

Read "housing_one_row_per_tenant.csv" with "," between the columns (guessed). To be sure, give separator=",".


explained difference – reference (C21)  housing (MASS) | explain (outcome="ordered groups", in_order=) | Infl (Low) estimate: simplydata -1.28881911108, R MASS polr (help page example) (MASS 7.3.66) -1.28882746182; relative difference 6.48e-06 (allowed 1e-06)
      why: R and lifelines references at their default stopping rules - https://github.com/bioforkgithub/simplydata_tests/blob/main/FINDINGS.md#c21-r-and-lifelines-references-at-their-default-stopping-rules
      Source: R MASS polr (help page example) (MASS 7.3.66) (run reference) - https://search.r-project.org/CRAN/refmans/MASS/html/polr.html
explained difference – reference (C21)  housing (MASS) | explain (outcome="ordered groups", in_order=) | Infl (Medium) estimate: simplydata -0.722425373231, R MASS polr (help page example) (MASS 7.3.66) -0.722437761733; relative difference 1.71e-05 (allowed 1e-06)
      why: R and lifelines references at their default stopping rules - https://github.com/bioforkgithub/simplydata_tests/blob/m

## 8. Percentile lines (quantile regression): Engel's food expenditure

The quantreg vignette prints the median regression (81.4822474 + 0.5601806 income; `se = "nid"` standard errors
19.25066 and 0.02828); the other percentiles against the help-page call of `rq` run once.

In [12]:
q = quiet(sdata.table(cmp.data_file("engel")).explain, "foodexp", using=["income"], for_percentile=[5, 25, 50, 75, 95])
for p in (5, 25, 50, 75, 95):
    s = Result(q, key="term", where={"percentile": p})
    for term in ("(intercept)", "income"):
        cmp.check("engel (quantreg)", "explain (for_percentile=)", f"percentile {p}: {term}", s[term, "estimate"])
        cmp.check("engel (quantreg)", "explain (for_percentile=)", f"percentile {p}: {term} standard error", s[term, "standard error"])

Read "engel.csv" with "," between the columns (guessed). To be sure, give separator=",".


agrees  engel (quantreg) | explain (for_percentile=) | percentile 5: (intercept): simplydata 124.880040813, R quantreg rq (help page example) (quantreg 6.1) 124.880040813; relative difference 1.14e-16 (allowed 1e-06)
      Source: R quantreg rq (help page example) (quantreg 6.1) (run reference) - https://search.r-project.org/CRAN/refmans/quantreg/html/rq.html
agrees  engel (quantreg) | explain (for_percentile=) | percentile 5: (intercept) standard error: simplydata 35.0667318136, R quantreg summary.rq (se = nid) (quantreg 6.1) 35.0667318136; relative difference 0.00e+00 (allowed 1e-06)
      Source: R quantreg summary.rq (se = nid) (quantreg 6.1) (run reference) - https://search.r-project.org/CRAN/refmans/quantreg/html/summary.rq.html
agrees  engel (quantreg) | explain (for_percentile=) | percentile 5: income: simplydata 0.343361057632, R quantreg rq (help page example) (quantreg 6.1) 0.343361057632; relative difference 1.62e-16 (allowed 1e-06)
      Source: R quantreg rq (help page ex

## 9. Giving outliers less weight (Huber's M-estimation): stack loss

The statsmodels `RLM` docstring prints the Huber fit of Brownlee's stack loss data to 8 decimals.

In [13]:
stackloss = sdata.table(DATA + "stackloss.csv")
COLS = ["air_flow", "water_temp", "acid_conc"]
rl = Result(quiet(stackloss.explain, "stack_loss", using=COLS, giving_outliers_less_weight=True), key="term")
for term in ["(intercept)"] + COLS:
    cmp.check("stackloss", "explain (giving_outliers_less_weight=True)", f"{term} estimate", rl[term, "effect"])
    cmp.check("stackloss", "explain (giving_outliers_less_weight=True)", f"{term} standard error", rl[term, "standard error"])

Read "stackloss.csv" with "," between the columns (guessed). To be sure, give separator=",".
agrees  stackloss | explain (giving_outliers_less_weight=True) | (intercept) estimate: simplydata -41.0264983524, statsmodels RLM docstring (HuberT) -41.02649835; difference 2.40e-09 (allowed 5e-09)
      Source: statsmodels RLM docstring (HuberT) (printed) - https://www.statsmodels.org/v0.15.0/generated/statsmodels.robust.robust_linear_model.RLM.html
agrees  stackloss | explain (giving_outliers_less_weight=True) | (intercept) standard error: simplydata 9.79189854136, statsmodels RLM docstring (HuberT) 9.79189854; difference 1.36e-09 (allowed 5e-09)
      Source: statsmodels RLM docstring (HuberT) (printed) - https://www.statsmodels.org/v0.15.0/generated/statsmodels.robust.robust_linear_model.RLM.html
agrees  stackloss | explain (giving_outliers_less_weight=True) | air_flow estimate: simplydata 0.8293843346, statsmodels RLM docstring (HuberT) 0.82938433; difference 4.60e-09 (allowed 5e-09)
    

## 10. Standardised effects and precision weights

`show="standardised effects"` against R `effectsize::standardize_parameters(method = "refit")`; `weights="beetles"`
(the share of beetles killed, each dose weighted by its number of beetles) against R `lm(weights =)`.

In [14]:
st = Result(quiet(stackloss.explain, "stack_loss", using=COLS, show="standardised effects"), key="term")
cmp.check_all("stackloss", 'explain (show="standardised effects")',
              {**{f"{t} estimate": st[t, "effect"] for t in COLS}, **{f"{t} 95 % low": st[t, "95 % CI low"] for t in COLS}})
quiet(beetles.add_column, "share", "killed / beetles")
w = Result(quiet(beetles.explain, "share", using=["log10_dose"], weights="beetles"), key="term")
cmp.check_all("Bliss beetles", "explain (weights=)",
              {"intercept": w["(intercept)", "effect"], "slope": w["log10_dose", "effect"],
               "slope standard error": w["log10_dose", "standard error"], "slope p-value": w["log10_dose", "p-value"]})

agrees  stackloss | explain (show="standardised effects") | air_flow estimate: simplydata 0.645047662682, R effectsize standardize_parameters (refit) (effectsize 1.0.3) 0.645047662682; relative difference 6.88e-16 (allowed 1e-10)
      Source: R effectsize standardize_parameters (refit) (effectsize 1.0.3) (run reference) - https://search.r-project.org/CRAN/refmans/effectsize/html/standardize_parameters.html
agrees  stackloss | explain (show="standardised effects") | water_temp estimate: simplydata 0.402502491464, R effectsize standardize_parameters (refit) (effectsize 1.0.3) 0.402502491464; relative difference 5.52e-16 (allowed 1e-10)
      Source: R effectsize standardize_parameters (refit) (effectsize 1.0.3) (run reference) - https://search.r-project.org/CRAN/refmans/effectsize/html/standardize_parameters.html
agrees  stackloss | explain (show="standardised effects") | acid_conc estimate: simplydata -0.0801405433495, R effectsize standardize_parameters (refit) (effectsize 1.0.3) -0.0

## 11. Predicted values and their intervals: `add_predicted_column`

Against R `predict()`: the confidence and prediction intervals of a linear model (stackloss), the intervals of a
logistic (birthwt) and a Poisson model (NMES 1988; every 100th of its 4406 rows) on the link scale turned back,
and a weighted formula fit.

In [15]:
for new_rows in (False, True):
    table = sdata.table(DATA + "stackloss.csv")
    quiet(table.add_predicted_column, "stack_loss", using=["air_flow", "water_temp"], interval=95, for_new_rows=new_rows)
    word = "prediction" if new_rows else "CI"
    t = Result(table)
    check = f"add_predicted_column (interval=95{', for_new_rows=True' if new_rows else ''})"
    cmp.check_vector("stackloss", check, "predicted", t["predicted stack_loss"])
    cmp.check_vector("stackloss", check, "low", t[f"predicted stack_loss 95 % {word} low"])
    cmp.check_vector("stackloss", check, "high", t[f"predicted stack_loss 95 % {word} high"])

births = sdata.table(FETCHED + "birthwt.csv", separator=",")
quiet(births.add_predicted_column, "low", using=["age", "lwt", "smoke"], outcome="yes or no", interval=95)
t = Result(births)
for key, column in (("predicted", "predicted low"), ("low", "predicted low 95 % CI low"), ("high", "predicted low 95 % CI high")):
    cmp.check_vector("birthwt", 'add_predicted_column (outcome="yes or no", interval=95)', key, t[column])

visits = sdata.table(FETCHED + "nmes1988.csv", separator=",")
quiet(visits.add_predicted_column, "visits", using=["hospital", "chronic"], outcome="counts", interval=95)
t = Result(visits)
for key, column in (("predicted", "predicted visits"), ("low", "predicted visits 95 % CI low"), ("high", "predicted visits 95 % CI high")):
    cmp.check_vector("NMES 1988", 'add_predicted_column (outcome="counts", interval=95)', key, t[column])

quiet(beetles.add_predicted_column, "share = a + b * log10_dose", weights="beetles")
t = Result(beetles)
cmp.check_vector("Bliss beetles", "add_predicted_column (formula, weights=)", "predicted",
                 t[[c for c in t.columns if c.startswith("predicted")][0]])

Read "stackloss.csv" with "," between the columns (guessed). To be sure, give separator=",".
agrees  stackloss | add_predicted_column (interval=95) | predicted [1]: simplydata 38.3080021357, R stats predict.lm (stats 4.6.2) 38.3080021357; relative difference 1.85e-16 (allowed 1e-10)
      Source: R stats predict.lm (stats 4.6.2) (run reference) - https://search.r-project.org/CRAN/refmans/stats/html/predict.lm.html
agrees  stackloss | add_predicted_column (interval=95) | predicted [2]: simplydata 38.3080021357, R stats predict.lm (stats 4.6.2) 38.3080021357; relative difference 1.85e-16 (allowed 1e-10)
      Source: R stats predict.lm (stats 4.6.2) (run reference) - https://search.r-project.org/CRAN/refmans/stats/html/predict.lm.html
agrees  stackloss | add_predicted_column (interval=95) | predicted [3]: simplydata 32.3615271951, R stats predict.lm (stats 4.6.2) 32.3615271951; relative difference 0.00e+00 (allowed 1e-10)
      Source: R stats predict.lm (stats 4.6.2) (run reference) - h

## 12. Rows that pull the model, and the numbers of `check_model_of`

`show_rows_that_pull_model` gives back a table of the rows with a Cook's distance above 4 / n, the biggest first: the
row's number, its Cook's distance, its leverage and its studentised residual (a straight-line model) or standardised
Pearson residual (a yes/no model). Compared with R `cooks.distance`, `hatvalues` and `rstudent` /
`rstandard(type = "pearson")` on the model of R's `influence.measures` help page (LifeCycleSavings) and on the
logistic model of section 11 (birthwt). `check_model_of(..., show="numbers")` gives back the numbers it prints:
the variance inflation factors (R `car::vif`), the Breusch-Pagan test in Koenker's studentised form and White's test
(R `lmtest::bptest`, on the columns and on the columns, their squares and products) and the count of rows with a
Cook's distance above 0.5.

In [16]:
lcs = sdata.table(cmp.data_file("LifeCycleSavings"))
births = sdata.table(FETCHED + "birthwt.csv", separator=",")
for data, dataset, outcome, using, column, residual in (
        (lcs, "LifeCycleSavings", "numbers", ["pop15", "pop75", "dpi", "ddpi"], "sr", "studentised residual"),
        (births, "birthwt", "yes or no", ["age", "lwt", "smoke"], "low", "standardised residual")):
    pulled = Result(quiet(data.show_rows_that_pull_model, column, using=using, outcome=outcome))
    for name in ("row", "Cook's distance", "leverage", residual):
        cmp.check_vector(dataset, f'show_rows_that_pull_model (outcome="{outcome}")', name, pulled[name])

numbers = Result(quiet(lcs.check_model_of, "sr", using=["pop15", "pop75", "dpi", "ddpi"], show="numbers"))
found = {}
for i in range(1, len(numbers) + 1):
    row = numbers.row(i)
    name = f"{row['check']}, {row['term']}" if row["term"] else row["check"]
    found[name] = row["value"]
    if row["check"].endswith("(chi-square)"):
        found[f"{name} df"] = row["df"]
        found[f"{name} p-value"] = row["p-value"]
cmp.check_all("LifeCycleSavings", 'check_model_of (show="numbers")', found)

Read "LifeCycleSavings.csv" with "," between the columns (guessed). To be sure, give separator=",".
agrees  LifeCycleSavings | show_rows_that_pull_model (outcome="numbers") | row [1]: simplydata 49, R stats cooks.distance, hatvalues, rstudent (stats 4.6.2) 49; difference 0.00e+00 (allowed 0e+00)
      Source: R stats cooks.distance, hatvalues, rstudent (stats 4.6.2) (run reference) - https://search.r-project.org/CRAN/refmans/stats/html/influence.measures.html
agrees  LifeCycleSavings | show_rows_that_pull_model (outcome="numbers") | row [2]: simplydata 23, R stats cooks.distance, hatvalues, rstudent (stats 4.6.2) 23; difference 0.00e+00 (allowed 0e+00)
      Source: R stats cooks.distance, hatvalues, rstudent (stats 4.6.2) (run reference) - https://search.r-project.org/CRAN/refmans/stats/html/influence.measures.html
agrees  LifeCycleSavings | show_rows_that_pull_model (outcome="numbers") | row [3]: simplydata 46, R stats cooks.distance, hatvalues, rstudent (stats 4.6.2) 46; difference 

In [17]:
cmp.finish()

25_models_outcomes: 1177 comparisons - 1162 agrees, 15 explained difference – reference
of the SKIPs, 0 are waiting for a simplydata word (see the reason column)


,notebook,dataset,method,quantity,simplydata,reference,reference source,difference,tolerance,tolerance type,score,status,reason,result,finding,explanation,link,kind,url,ran on
0,25_models_outcomes,NMES 1988,"explain (outcome=""counts"")",(intercept) estimate,1.028874,1.028874,"Zeileis, Kleiber & Jackman (2008), J. Stat. So...",1.950798e-07,5.000000e-07,absolute,1.950798e-07,PASS,,agrees,,,,printed,https://doi.org/10.18637/jss.v027.i08 ; https:...,"GitHub Actions, Linux 7.0.0-1020-azure, Intel(..."
1,25_models_outcomes,NMES 1988,"explain (outcome=""counts"")",(intercept) standard error,0.023785,0.023785,"Zeileis, Kleiber & Jackman (2008), J. Stat. So...",-1.087381e-07,5.000000e-07,absolute,1.087381e-07,PASS,,agrees,,,,printed,https://doi.org/10.18637/jss.v027.i08 ; https:...,"GitHub Actions, Linux 7.0.0-1020-azure, Intel(..."
2,25_models_outcomes,NMES 1988,"explain (outcome=""counts"")",hospital estimate,0.164797,0.164797,"Zeileis, Kleiber & Jackman (2008), J. Stat. So...",3.892092e-07,5.000000e-07,absolute,3.892092e-07,PASS,,agrees,,,,printed,https://doi.org/10.18637/jss.v027.i08 ; https:...,"GitHub Actions, Linux 7.0.0-1020-azure, Intel(..."
3,25_models_outcomes,NMES 1988,"explain (outcome=""counts"")",hospital standard error,0.005997,0.005997,"Zeileis, Kleiber & Jackman (2008), J. Stat. So...",3.909379e-07,5.000000e-07,absolute,3.909379e-07,PASS,,agrees,,,,printed,https://doi.org/10.18637/jss.v027.i08 ; https:...,"GitHub Actions, Linux 7.0.0-1020-azure, Intel(..."
4,25_models_outcomes,NMES 1988,"explain (outcome=""counts"")",health (poor) estimate,0.248307,0.248307,"Zeileis, Kleiber & Jackman (2008), J. Stat. So...",-2.861368e-08,5.000000e-07,absolute,2.861368e-08,PASS,,agrees,,,,printed,https://doi.org/10.18637/jss.v027.i08 ; https:...,"GitHub Actions, Linux 7.0.0-1020-azure, Intel(..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1172,25_models_outcomes,LifeCycleSavings,"check_model_of (show=""numbers"")",Breusch-Pagan test (chi-square) p-value,0.288823,0.288823,R lmtest bptest (studentised) (lmtest 0.9.40),-3.330669e-16,1.000000e-10,relative,1.153185e-15,PASS,,agrees,,,,run reference,https://search.r-project.org/CRAN/refmans/lmte...,"GitHub Actions, Linux 7.0.0-1020-azure, Intel(..."
1173,25_models_outcomes,LifeCycleSavings,"check_model_of (show=""numbers"")",White's test (chi-square),13.910971,13.910971,R lmtest bptest (studentised) (lmtest 0.9.40),2.131628e-14,1.000000e-10,relative,1.532336e-15,PASS,,agrees,,,,run reference,https://search.r-project.org/CRAN/refmans/lmte...,"GitHub Actions, Linux 7.0.0-1020-azure, Intel(..."
1174,25_models_outcomes,LifeCycleSavings,"check_model_of (show=""numbers"")",White's test (chi-square) df,14.000000,14.000000,R lmtest bptest (studentised) (lmtest 0.9.40),0.000000e+00,1.000000e-10,relative,0.000000e+00,PASS,,agrees,,,,run reference,https://search.r-project.org/CRAN/refmans/lmte...,"GitHub Actions, Linux 7.0.0-1020-azure, Intel(..."
1175,25_models_outcomes,LifeCycleSavings,"check_model_of (show=""numbers"")",White's test (chi-square) p-value,0.456365,0.456365,R lmtest bptest (studentised) (lmtest 0.9.40),-1.498801e-15,1.000000e-10,relative,3.284218e-15,PASS,,agrees,,,,run reference,https://search.r-project.org/CRAN/refmans/lmte...,"GitHub Actions, Linux 7.0.0-1020-azure, Intel(..."
